# Dataset & Scope 

MIMIC-IV is a large, de-identified electronic health record database containing data from hospital admissions of 364,627 unique patients admitted to the Beth Israel Deaconess Medical Center. It is organized into two main modules: the Hospital module, covering the full hospitalization, and the ICU module, covering intensive care unit stays specifically.

Our objective is to build a vancomycin dosing recommendation system that, given a patient's dosing history, blood concentration measurements, and clinical covariates (weight, age, creatinine), infers individual pharmacokinetic parameters (clearance CL and volume of distribution Vd), then recommends the optimal next dose to maintain concentrations within the therapeutic window.

### Extraction Pipeline

When a patient comes to the hospital, MIMIC creates records at three nested levels:

subject_id  →  one real human being (can come back many times)

  └── hadm_id  →  one hospital admission (one front-door visit)
  
        └── stay_id  →  one ICU stay within that admission


A single patient (subject_id) can have 5 hospital admissions over their life (hadm_id). Within one admission they might go to the ICU twice (stay_id). These are genuinely different events — different dates, different clinical states, different vancomycin courses.

In [1]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import duckdb
con = duckdb.connect()

In [2]:
duckdb.query("""
    CREATE OR REPLACE VIEW labs        AS SELECT * FROM read_csv_auto('/kaggle/input/datasets/oumaymabejaoui/mimiciv-31/hosp/labevents.csv');
    CREATE OR REPLACE VIEW chartevents AS SELECT * FROM read_csv_auto('/kaggle/input/datasets/oumaymabejaoui/mimiciv-31/icu/chartevents.csv');
    CREATE OR REPLACE VIEW icustays    AS SELECT * FROM read_csv_auto('/kaggle/input/datasets/oumaymabejaoui/mimiciv-31/icu/icustays.csv');
    CREATE OR REPLACE VIEW inputevents AS SELECT * FROM read_csv_auto('/kaggle/input/datasets/oumaymabejaoui/mimiciv-31/icu/inputevents.csv');
    CREATE OR REPLACE VIEW prescriptions AS SELECT * FROM read_csv_auto('/kaggle/input/datasets/oumaymabejaoui/mimiciv-31/hosp/prescriptions.csv');
    CREATE OR REPLACE VIEW patients    AS SELECT * FROM read_csv_auto('/kaggle/input/datasets/oumaymabejaoui/mimiciv-31/hosp/patients.csv');
    CREATE OR REPLACE VIEW procedureevents  AS SELECT * FROM read_csv_auto('/kaggle/input/datasets/oumaymabejaoui/mimiciv-31/icu/procedureevents.csv');
    CREATE OR REPLACE VIEW outputevents  AS SELECT * FROM read_csv_auto('/kaggle/input/datasets/oumaymabejaoui/mimiciv-31/icu/outputevents.csv');

""")

# 1. item ID lookup

MIMIC doesn't store drug names or measurement names as strings in the big tables — it stores them as integer codes called itemid. d_items and d_labitems are the dictionaries that translate those integer codes into human-readable labels

In [3]:
d_items= pd.read_csv("/kaggle/input/datasets/oumaymabejaoui/mimiciv-31/icu/d_items.csv")
d_labitems=pd.read_csv("/kaggle/input/datasets/oumaymabejaoui/mimiciv-31/hosp/d_labitems.csv")

print("d_items columns    :", d_items.columns.tolist())
print("d_labitems columns :", d_labitems.columns.tolist())

d_items columns    : ['itemid', 'label', 'abbreviation', 'linksto', 'category', 'unitname', 'param_type', 'lownormalvalue', 'highnormalvalue']
d_labitems columns : ['itemid', 'label', 'fluid', 'category']


****

## 1.1 Vancomycin ID lookup

In [4]:
# 'Vancomycin' id (inputevents )
print(d_items[d_items['label'] == 'Vancomycin']['itemid'].to_string(index=False))

225798


In [5]:
VANCO_DOSE_ITEMIDS  = (225798)                   # IV vancomycin dose only

****

## 1.2 weight ID lookup

In [6]:
# 'weight' id (chartevents)
print(d_items[d_items['label'] == 'Admission Weight (Kg)']['itemid'].to_string(index=False))

226512


In [7]:
WEIGHT_ITEMIDS      = (226512)            # admission weight 

****

## 1.3 height ID lookup

In [8]:
# 'height' id (cm (chartevents)
print(d_items[d_items['label'] == 'Height (cm)']['itemid'].to_string(index=False))

226730


In [9]:
HEIGHT_ITEMIDS      = (226730)                   # height in cm

****

## 1.4 Creatinine ID lookup

In [10]:
# 'Creatinine' id (labevents)
print(d_labitems[d_labitems['label'].str.contains('creatinine', case=False, na=False)][['itemid', 'label', 'fluid', 'category']]
      .to_string(index=False))

 itemid                           label               fluid  category
  50841             Creatinine, Ascites             Ascites Chemistry
  50912                      Creatinine               Blood Chemistry
  51021         Creatinine, Joint Fluid         Joint Fluid Chemistry
  51032          Creatinine, Body Fluid    Other Body Fluid Chemistry
  51052             Creatinine, Pleural             Pleural Chemistry
  51067                24 hr Creatinine               Urine Chemistry
  51070       Albumin/Creatinine, Urine               Urine Chemistry
  51073 Amylase/Creatinine Ratio, Urine               Urine Chemistry
  51080            Creatinine Clearance               Urine Chemistry
  51081               Creatinine, Serum               Urine Chemistry
  51082               Creatinine, Urine               Urine Chemistry
  51099        Protein/Creatinine Ratio               Urine Chemistry
  51106                Urine Creatinine               Urine Chemistry
  51787             

In [11]:
CREATININE_ITEMIDS  = (50912, 52546)              # blood creatinine only

****

## 1.5 bun ID lookup

In [12]:
# 'BUN' id
print(d_labitems[d_labitems['label'].str.contains('urea|BUN', case=False, na=False)
][['itemid', 'label', 'fluid', 'category']].to_string())

      itemid                       label                fluid   category
49     50851      Urea Nitrogen, Ascites              Ascites  Chemistry
202    51006               Urea Nitrogen                Blood  Chemistry
233    51045   Urea Nitrogen, Body Fluid     Other Body Fluid  Chemistry
292    51104        Urea Nitrogen, Urine                Urine  Chemistry
920    51804          Urea Nitrogen, CSF  Cerebrospinal Fluid  Chemistry
941    51825  Urea Nitrogen, Joint Fluid          Joint Fluid  Chemistry
957    51842                         Bun     Other Body Fluid  Chemistry
1013   51922      Urea Nitrogen, Pleural              Pleural  Chemistry
1042   51951        Urea Nitrogen, Stool                Stool  Chemistry
1529   52647               Urea Nitrogen                Blood  Chemistry


In [13]:
BUN_ITEMIDS         = (51006, 52647)              # blood urea nitrogen only

****

## 1.6 albumin ID lookup

In [14]:
# 'Albumin' id
print(d_labitems[d_labitems['label'].str.contains('albumin', case=False, na=False)
][['itemid', 'label', 'fluid', 'category']].to_string())

      itemid                      label                fluid   category
33     50835           Albumin, Ascites              Ascites  Chemistry
60     50862                    Albumin                Blood  Chemistry
207    51019       Albumin, Joint Fluid          Joint Fluid  Chemistry
213    51025        Albumin, Body Fluid     Other Body Fluid  Chemistry
234    51046           Albumin, Pleural              Pleural  Chemistry
257    51069             Albumin, Urine                Urine  Chemistry
258    51070  Albumin/Creatinine, Urine                Urine  Chemistry
891    51775                  (Albumin)  Cerebrospinal Fluid  Chemistry
892    51776                  <Albumin>  Cerebrospinal Fluid  Chemistry
893    51777               Albumin, CSF  Cerebrospinal Fluid  Chemistry
923    51807                  (Albumin)          Joint Fluid  Chemistry
951    51836                  <Albumin>     Other Body Fluid  Chemistry
990    51897         Surfactant/Albumin     Other Body Fluid  Ch

In [15]:
ALBUMIN_ITEMIDS     = (50862, 52022, 53085, 53138) # blood albumin only

****

## 1.6 alt ID lookup

In [16]:
# 'alt' id
print(d_labitems[d_labitems['label'].str.contains('alt', case=False, na=False)
][['itemid', 'label', 'fluid', 'category']].to_string())

    itemid                           label  fluid   category
59   50861  Alanine Aminotransferase (ALT)  Blood  Chemistry


In [17]:
ALT_ITEMIDS     = (50861)

****

## 1.7 ast ID lookup

In [18]:
# 'ast' id
print(d_labitems[d_labitems['label'].str.contains(r'\bAST\b', case=False, na=False)]
      [['itemid', 'label', 'fluid', 'category']]
      .to_string(index=False))

 itemid                           label fluid  category
  50878 Asparate Aminotransferase (AST) Blood Chemistry


In [19]:
AST_ITEMIDS     = (50878)

****

## 1.8 vasopressor ID lookup

In [20]:
# 'vasopressor' id
vasopressor_keywords = 'Norepinephrine|Epinephrine|Dopamine|Vasopressin|Phenylephrine'
print(d_items[d_items['label'].str.contains(vasopressor_keywords, case=False, na=False)]
      [['itemid', 'label', 'unitname', 'category']].to_string(index=False))

 itemid                         label unitname    category
 221289                   Epinephrine       mg Medications
 221662                      Dopamine       mg Medications
 221749                 Phenylephrine       mg Medications
 221906                Norepinephrine       mg Medications
 222315                   Vasopressin    units Medications
 229617                  Epinephrine.       mg Medications
 229630        Phenylephrine (50/250)       mg Medications
 229631 Phenylephrine (200/250)_OLD_1       mg Medications
 229632       Phenylephrine (200/250)       mg Medications
 229789    Phenylephrine (Intubation)      mcg  Intubation


In [21]:
VASOPRESSOR_ITEMIDS = (221906,                    # norepinephrine
                       221289,                    # epinephrine
                       221662,                    # dopamine
                       222315,                    # vasopressin
                       221749)                    # phenylephrine

Vasopressors are drugs given to patients whose blood pressure has dropped dangerously low — they constrict the blood vessels to push the pressure back up.
If a patient has_vasopressor = 1, it means they were in septic shock or cardiovascular collapse at some point during their stay. It's a marker of how critically ill they were.

****

## 1.9 machanical ventillation ID lookup

In [22]:
# 'MECH_VENT' id
print(d_items[d_items['label'] == 'Invasive Ventilation']['itemid'].to_string(index=False))
print(d_items[d_items['label'] == 'Non-invasive Ventilation']['itemid'].to_string(index=False))

225792
225794


In [23]:
MECH_VENT_ITEMIDS   = (225792,                    # invasive ventilation
                       225794)                    # non-invasive ventilation

"""
has_invasive-ventilation
has_non_invasive_ventillation
"""

'\nhas_invasive-ventilation\nhas_non_invasive_ventillation\n'

****

## 1.10 dialysis ID lookup

In [24]:
# 'dialysis' id
print(d_items[d_items['label'].str.contains('dialysis|CRRT|hemodialysis|RRT', case=False, na=False)]
      [['itemid', 'label', 'unitname', 'category']].to_string(index=False))

 itemid                                             label unitname                category
 224135                              Dialysis Access Site      NaN                Dialysis
 224139                          Dialysis Site Appearance      NaN                Dialysis
 224270                                 Dialysis Catheter      NaN Access Lines - Invasive
 225126                                  Dialysis patient      NaN        Adm History/FHPA
 225128                                     Last dialysis      NaN        Adm History/FHPA
 225318                      Dialysis Catheter Cap Change      NaN Access Lines - Invasive
 225319           Dialysis Catheter Change over Wire Date      NaN Access Lines - Invasive
 225321                 Dialysis Catheter Dressing Change      NaN Access Lines - Invasive
 225322                  Dialysis Catheter Insertion Date      NaN Access Lines - Invasive
 225323                     Dialysis Catheter Site Appear      NaN Access Lines - Invasive

In [25]:
DIALYSIS_ITEMIDS    = (225441,                    # hemodialysis
                       225802,                    # CRRT
                       225803,                    # CVVHD
                       225809,                    # CVVHDF
                       225805,                    # peritoneal dialysis
                       225955)                    # SCUF

""""
has_hd
has_crrt
has_pd
has_other_dialysis
"""

'"\nhas_hd\nhas_crrt\nhas_pd\nhas_other_dialysis\n'

****

## 1.11 ID lookup for Vancomycin levels

In [26]:
# Search d_labitems for blood level monitoring
print(d_labitems[d_labitems['label'].str.contains('vancomycin', case=False, na=False)]
      [['itemid', 'label', 'fluid', 'category']]
      .to_string(index=False))

 itemid      label fluid  category
  51009 Vancomycin Blood Chemistry


In [27]:
query1 = """
    SELECT itemid, label, abbreviation, category
    FROM d_items
    WHERE (label ILIKE '%vancomycin%' OR abbreviation ILIKE '%vancomycin%')
      AND category = 'Labs'
    ORDER BY label;
"""

vanco_icu_labels = duckdb.query(query1).df()
print(vanco_icu_labels.to_string(index=False))

 itemid                label         abbreviation category
 227453    Vancomycin (Peak)    Vancomycin (Peak)     Labs
 227454  Vancomycin (Random)  Vancomycin (Random)     Labs
 227455  Vancomycin (Trough)  Vancomycin (Trough)     Labs
 225697   ZVancomycin (Peak)   ZVancomycin (Peak)     Labs
 226065 ZVancomycin (Random) ZVancomycin (Random)     Labs
 226064 ZVancomycin (Trough) ZVancomycin (Trough)     Labs


 * 227453    Vancomycin (Peak)  :  Measured shortly after infusion ends
 *  227454  Vancomycin (Random) : Measured at an arbitrary time
 *  227455  Vancomycin (Trough)  : Measured right before the next dose

**debatable**

## 1.11.0 ARE the Vancomycin and the ZVancomycin variables both present?

In [28]:
query = """
SELECT
    itemid,
    COUNT(*) AS n_records
FROM chartevents
WHERE itemid IN (
    227453,227454,227455,
    225697,226065,226064
)
GROUP BY itemid
ORDER BY itemid
"""

df = duckdb.query(query).df()

print(df)

vanco = df[df.itemid.isin([227453,227454,227455])]["n_records"].sum()
zvanco = df[df.itemid.isin([225697,226065,226064])]["n_records"].sum()

print("\nSUMMARY")
print(f"Vancomycin observations  : {vanco:,}")
print(f"ZVancomycin observations : {zvanco:,}")

if zvanco == 0:
    print("\nNo ZVancomycin observations found.")
    print("Statistical comparison cannot be performed.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   itemid  n_records
0  227453         18
1  227454      20020
2  227455      20092

SUMMARY
Vancomycin observations  : 40,130
ZVancomycin observations : 0

No ZVancomycin observations found.
Statistical comparison cannot be performed.


from chartevents  (227455, 227454, 227453) and 51009 from labevents.

In MIMIC-III Clinical Database, the difference between labevents and chartevents is fundamentally about source system and clinical workflow.
* origin of vancomycin labevents: Central laboratory system
* origin of vancomycin chartevents: Nurse/bedside documentation

In [29]:
VANCO_LEVEL_ITEMIDS = (227455, 227453 , 227454)

In [30]:
VANCO_DOSE_ITEMIDS     = (225798,)
VANCO_LEVEL_ITEMIDS    = (227455, 227453, 227454)   # trough, peak, random — labevents only
VANCO_LAB_ITEMID       = (51009,)
WEIGHT_ITEMIDS         = (226512,)
HEIGHT_ITEMIDS         = (226730,)
CREATININE_ITEMIDS     = (50912, 52546)
BUN_ITEMIDS            = (51006, 52647)
ALBUMIN_ITEMIDS        = (50862, 52022, 53085, 53138)
ALT_ITEMIDS            = (50861,)
AST_ITEMIDS            = (50878,)
VASOPRESSOR_ITEMIDS    = (221906,   # norepinephrine
                          221289,   # epinephrine
                          221662,   # dopamine
                          222315,   # vasopressin
                          221749)   # phenylephrine
MECH_VENT_ITEMIDS      = (225792,   # invasive
                          225794)   # non-invasive
DIALYSIS_ITEMIDS       = (225441,   # hemodialysis
                          225802,   # CRRT
                          225803,   # CVVHD
                          225809,   # CVVHDF
                          225805,   # peritoneal
                          225955)   # SCUF

****

****

# 2. concentration extraction  for Vancomycin

The Real Problem — Chartevents is often a copy

Blood draw
    ↓
Lab analyzes it  →  labevents: valuenum = 14.3  ✓ (ground truth)
    ↓
Nurse sees result, types it into the chart
    →  chartevents: valuenum = 14.0  ✗ (rounded, or typo)


The nurse is re-entering what the lab already measured. So you now have two records of the same measurement, but the lab one is the original and the chart one is a human copy — potentially rounded, slightly wrong, or entered at a slightly different time.

In [31]:
query2 = duckdb.query("""
    WITH set_a AS (
        /* Stays from chartevents */
        SELECT DISTINCT stay_id FROM chartevents 
        WHERE itemid IN (227455, 227453 , 227454)
        AND stay_id IS NOT NULL
    ),
    set_b AS (
        /* Stays from labevents - Joined with icustays to get stay_id */
        SELECT DISTINCT i.stay_id 
        FROM labs l
        JOIN icustays i ON l.hadm_id = i.hadm_id
        WHERE l.itemid = 51009
        AND i.stay_id IS NOT NULL
    )
    SELECT 
        (SELECT COUNT(*) FROM set_a WHERE stay_id NOT IN (SELECT stay_id FROM set_b)) AS only_A,
        (SELECT COUNT(*) FROM set_b WHERE stay_id NOT IN (SELECT stay_id FROM set_a)) AS only_B,
        (SELECT COUNT(*) FROM set_a WHERE stay_id IN (SELECT stay_id FROM set_b)) AS both_intersect
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [32]:
print(query2.iloc[0])

only_A               75
only_B             8261
both_intersect    15691
Name: 0, dtype: int64


In [33]:
query_lab = """
    SELECT 
        i.stay_id,
        epoch(l.charttime - i.intime)/3600 AS time_hours,
        l.valuenum,
        1 AS event_type
    FROM labs l
    JOIN icustays i ON l.hadm_id = i.hadm_id
    WHERE l.itemid = 51009
"""

df_lab = duckdb.query(query_lab).df()
print(f"Unique stays: {df_lab['stay_id'].nunique()}")
print(f"Total events: {len(df_lab)}")
print(df_lab.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Unique stays: 23952
Total events: 83474
    stay_id  time_hours  valuenum  event_type
0  37081114  131.800000      28.2           1
1  37081114  202.500000      20.9           1
2  37067082   29.949444       8.0           1
3  37067082   44.532778      11.9           1
4  34592300   61.610000      11.1           1


In [34]:
# 1. How many negative time_hours do you have?
print(df_lab[df_lab['time_hours'] < 0]['stay_id'].nunique())

# 2. Any nulls?
print(df_lab.isnull().sum())

4877
stay_id         0
time_hours      0
valuenum      581
event_type      0
dtype: int64


## 2.1 Take ALL labevents, Find the 75 chartevents-only stays and add them in order to Build a concentration events for vanco

### event_type 1: concentrations of vamocin

In [35]:
query_75 = """
    WITH lab_stays AS (
        SELECT DISTINCT i.stay_id
        FROM labs l
        JOIN icustays i ON l.hadm_id = i.hadm_id
        WHERE l.itemid = 51009
    )
    SELECT 
        i.stay_id,
        epoch(c.charttime - i.intime)/3600 AS time_hours,
        c.valuenum,
        1 AS event_type
    FROM chartevents c
    JOIN icustays i ON c.stay_id = i.stay_id
    WHERE c.itemid IN (227455, 227454, 227453, 226064, 226065, 225697)
      AND c.stay_id NOT IN (SELECT stay_id FROM lab_stays)
      AND c.valuenum IS NOT NULL
"""

df_patch = duckdb.query(query_75).df()
print(f"Unique stays: {df_patch['stay_id'].nunique()}")
print(f"Total events: {len(df_patch)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Unique stays: 75
Total events: 201


In [36]:
df_concentration = pd.concat([df_lab, df_patch], ignore_index=True)

print(f"Unique stays: {df_concentration['stay_id'].nunique()}")
print(f"Total events: {len(df_concentration)}")
print(df_concentration.head())

Unique stays: 24027
Total events: 83675
    stay_id  time_hours  valuenum  event_type
0  37081114  131.800000      28.2           1
1  37081114  202.500000      20.9           1
2  37067082   29.949444       8.0           1
3  37067082   44.532778      11.9           1
4  34592300   61.610000      11.1           1


# 3. dose extraction

The goal : Extract two things for each vancomycin IV administration: dose in mg and precise administration time.

| | `prescriptions` | `inputevents` |
|---|---|---|
| **What it records** | The order written by the physician | The infusion logged by the ICU pump |
| **Timing** | When the order was activated | When the pump actually started |
| **Dose** | ✅ dose in mg, directly available | ❌ 97% of rows store "1 dose", not mg |
| **Coverage** | All inpatients | ICU patients only |

## 3.1 Understand each table independently

In [37]:
# ── Pull the filtered table ───────────────────────────────────────────────────
rx_vanco = duckdb.query("""
    SELECT subject_id, hadm_id, drug, route,
           dose_val_rx, dose_unit_rx, doses_per_24_hrs,
           starttime, stoptime
    FROM prescriptions
    WHERE drug ILIKE '%vancomycin%'
      AND route = 'IV'
      AND dose_unit_rx IS NOT NULL
      AND dose_val_rx  IS NOT NULL
""").df()

# ── Parse datetimes ───────────────────────────────────────────────────────────
rx_vanco['starttime'] = pd.to_datetime(rx_vanco['starttime'])
rx_vanco['stoptime']  = pd.to_datetime(rx_vanco['stoptime'])
rx_vanco['duration_hrs'] = (rx_vanco['stoptime'] - rx_vanco['starttime']).dt.total_seconds() / 3600

print(rx_vanco.head())
print(rx_vanco.describe().T)
sep = "=" * 60

# ── 1. Shape ──────────────────────────────────────────────────────────────────
print(sep)
print("1. SHAPE")
print(sep)
print(f"  Rows            : {len(rx_vanco):,}")
print(f"  Unique patients : {rx_vanco['subject_id'].nunique():,}")
print(f"  Unique admissions: {rx_vanco['hadm_id'].nunique():,}")

# ── 2. Missing values ─────────────────────────────────────────────────────────
print(f"\n{sep}")
print("2. MISSING VALUES")
print(sep)
cols = ['dose_val_rx','dose_unit_rx','doses_per_24_hrs','starttime','stoptime']
for c in cols:
    n_miss = rx_vanco[c].isna().sum()
    print(f"  {c:<20}: {n_miss:>6,}  ({100*n_miss/len(rx_vanco):.1f}%)")

# ── 3. Drug name variants ─────────────────────────────────────────────────────
print(f"\n{sep}")
print("3. DRUG NAME VARIANTS  (after IV + non-null filter)")
print(sep)
print(rx_vanco['drug'].value_counts().to_string())

# ── 4. Dose unit breakdown ────────────────────────────────────────────────────
print(f"\n{sep}")
print("4. DOSE UNIT BREAKDOWN")
print(sep)
print(rx_vanco['dose_unit_rx'].value_counts().to_string())

# ── 5. doses_per_24_hrs ───────────────────────────────────────────────────────
print(f"\n{sep}")
print("6. DOSES_PER_24_HRS")
print(sep)
d24 = rx_vanco['doses_per_24_hrs']
print(f"  missing: {d24.isna().sum():,}  ({100*d24.isna().mean():.1f}%)")
print(f"  min    : {d24.min()}")
print(f"  median : {d24.median()}")
print(f"  max    : {d24.max()}")
print(f"\n  Value counts:")
print(rx_vanco['doses_per_24_hrs'].value_counts().sort_index().to_string())

# ── 6. Prescription duration ──────────────────────────────────────────────────
print(f"\n{sep}")
print("7. PRESCRIPTION DURATION  (stoptime - starttime, hours)")
print(sep)
d = rx_vanco['duration_hrs']
n_neg = (d < 0).sum()
n_zero = (d == 0).sum()
print(f"  negative durations : {n_neg:,}   ← stoptime before starttime (data issue)")
print(f"  zero durations     : {n_zero:,}   ← same start and stop")
print(f"  min    : {d.min():.1f} hrs")
print(f"  p5     : {d.quantile(.05):.1f} hrs")
print(f"  median : {d.median():.1f} hrs")
print(f"  mean   : {d.mean():.1f} hrs")
print(f"  p95    : {d.quantile(.95):.1f} hrs")
print(f"  max    : {d.max():.1f} hrs")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   subject_id   hadm_id        drug route dose_val_rx dose_unit_rx  \
0    10000690  23280645  Vancomycin    IV        1000           mg   
1    10000690  23280645  Vancomycin    IV        1000           mg   
2    10000690  25860671  Vancomycin    IV        1000           mg   
3    10000690  25860671  Vancomycin    IV         750           mg   
4    10000690  25860671  Vancomycin    IV        1000           mg   

   doses_per_24_hrs           starttime            stoptime  duration_hrs  
0                 1 2150-09-21 08:00:00 2150-09-24 11:00:00          75.0  
1                 1 2150-09-24 08:00:00 2150-09-24 18:00:00          10.0  
2                 1 2150-11-09 08:00:00 2150-11-08 10:00:00         -22.0  
3                 1 2150-11-09 08:00:00 2150-11-12 18:00:00          82.0  
4                 1 2150-11-03 08:00:00 2150-11-08 10:00:00         122.0  
                     count                        mean                  min  \
subject_id        229616.0               149

In [38]:
ie_vanco = duckdb.query("""
    SELECT ie.subject_id, ie.hadm_id, ie.stay_id,
           ie.amount, ie.amountuom,
           ie.totalamountuom, ie.totalamountuom,
           ie.rate,   ie.rateuom,
           ie.starttime, ie.endtime,
           ie.ordercategoryname,
           ie.statusdescription,
           ie.patientweight
    FROM inputevents ie
    WHERE ie.itemid = 225798
""").df()

ie_vanco['starttime'] = pd.to_datetime(ie_vanco['starttime'])
ie_vanco['endtime']   = pd.to_datetime(ie_vanco['endtime'])
ie_vanco['duration_hrs'] = (
    ie_vanco['endtime'] - ie_vanco['starttime']
).dt.total_seconds() / 3600

print(ie_vanco.head())
print(ie_vanco.describe().T)

# ── 1. Shape ──────────────────────────────────────────────────────────────────
print(sep)
print("1. SHAPE")
print(sep)
print(f"  Rows             : {len(ie_vanco):,}")
print(f"  Unique patients  : {ie_vanco['subject_id'].nunique():,}")
print(f"  Unique admissions: {ie_vanco['hadm_id'].nunique():,}")
print(f"  Unique ICU stays : {ie_vanco['stay_id'].nunique():,}")

# ── 2. Missing values ─────────────────────────────────────────────────────────
print(f"\n{sep}")
print("2. MISSING VALUES")
print(sep)
cols = ['amount','amountuom','rate','rateuom','starttime','endtime','patientweight']
for c in cols:
    n_miss = ie_vanco[c].isna().sum()
    print(f"  {c:<15}: {n_miss:>7,}  ({100*n_miss/len(ie_vanco):.1f}%)")

# ── 3. Amount (dose count) breakdown ──────────────────────────────────────────
print(f"\n{sep}")
print("3. AMOUNT  (what is actually stored)")
print(sep)
print(f"  amountuom value counts:")
print(ie_vanco['amountuom'].value_counts().to_string())
print(f"\n  amount stats:")
d = ie_vanco['amount']
print(f"  min    : {d.min()}")
print(f"  median : {d.median()}")
print(f"  max    : {d.max()}")
print(f"\n  amount value counts (top 10):")
print(ie_vanco['amount'].value_counts().head(10).to_string())

# ── 4. Rate breakdown ─────────────────────────────────────────────────────────
print(f"\n{sep}")
print("4. RATE  (infusion rate)")
print(sep)
print(f"  rateuom value counts:")
print(ie_vanco['rateuom'].value_counts().to_string())

r = ie_vanco['rate'].dropna()
print(f"\n  rate stats (non-null only, n={len(r):,}):")
print(f"  min    : {r.min():.2f}")
print(f"  p5     : {r.quantile(.05):.2f}")
print(f"  p25    : {r.quantile(.25):.2f}")
print(f"  median : {r.median():.2f}")
print(f"  mean   : {r.mean():.2f}")
print(f"  p75    : {r.quantile(.75):.2f}")
print(f"  p95    : {r.quantile(.95):.2f}")
print(f"  max    : {r.max():.2f}")


# ── 5. Duration ───────────────────────────────────────────────────────────────
print(f"\n{sep}")
print("5. INFUSION DURATION  (endtime - starttime, hours)")
print(sep)
d = ie_vanco['duration_hrs']
n_neg  = (d < 0).sum()
n_zero = (d == 0).sum()
print(f"  negative durations : {n_neg:,}")
print(f"  zero durations     : {n_zero:,}")
print(f"  min    : {d.min():.2f} hrs")
print(f"  p5     : {d.quantile(.05):.2f} hrs")
print(f"  median : {d.median():.2f} hrs")
print(f"  mean   : {d.mean():.2f} hrs")
print(f"  p95    : {d.quantile(.95):.2f} hrs")
print(f"  max    : {d.max():.2f} hrs")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   subject_id   hadm_id   stay_id  amount amountuom totalamountuom  \
0    10000690  25860671  37081114     1.0      dose             mL   
1    10000690  25860671  37081114     1.0      dose             mL   
2    10000690  25860671  37081114     1.0      dose             mL   
3    10000690  25860671  37081114     1.0      dose             mL   
4    10000980  26913865  39765666     1.0      dose             mL   

  totalamountuom_1  rate rateuom           starttime             endtime  \
0               mL   NaN    None 2150-11-06 13:00:00 2150-11-06 13:01:00   
1               mL   NaN    None 2150-11-03 12:33:00 2150-11-03 12:34:00   
2               mL   NaN    None 2150-11-05 12:00:00 2150-11-05 12:01:00   
3               mL   NaN    None 2150-11-04 13:15:00 2150-11-04 13:16:00   
4               mL   NaN    None 2189-06-27 07:40:00 2189-06-27 07:41:00   

     ordercategoryname statusdescription  patientweight  duration_hrs  
0  08-Antibiotics (IV)   FinishedRunning          

* Finding in prescriptions: 229k rows, dose stored cleanly in dose_val_rx. Most units were mg (223k rows), the rest were non-numeric entries like "MD to order daily dose" that carried no usable value. Filtering to mg units gave a clean numeric dose for 97% of rows.
* Finding in inputevents: 140k rows for itemid = 225798. The dose field (amount) stored 1 with unit "dose" for 96.6% of rows — meaning the pump recorded that a dose ran, but not how many mg. Only 3.6k rows had an actual mg value. This table was nearly useless for dose extraction but had two things prescriptions lacked: precise pump timestamps and patient weight.

## 3.2 Check if the two tables overlap

In [39]:
# ── Pull both tables ───────────────────────────────────────────────────────
rx = duckdb.query("""
    SELECT subject_id, hadm_id,
           TRY_CAST(dose_val_rx AS DOUBLE) AS dose_mg,
           starttime   AS order_start,
           stoptime    AS order_stop
    FROM prescriptions
    WHERE drug        ILIKE '%vancomycin%'
      AND route       = 'IV'
      AND dose_unit_rx = 'mg'
      AND TRY_CAST(dose_val_rx AS DOUBLE) > 0
      AND starttime IS NOT NULL
      AND stoptime  IS NOT NULL
""").df()

ie = duckdb.query("""
    SELECT subject_id, hadm_id, stay_id,
           starttime AS admin_time,
           amount, amountuom,
           patientweight
    FROM inputevents
    WHERE itemid = 225798
      AND statusdescription = 'FinishedRunning'
      AND amount > 0
""").df()

rx['order_start'] = pd.to_datetime(rx['order_start'])
rx['order_stop']  = pd.to_datetime(rx['order_stop'])
ie['admin_time']  = pd.to_datetime(ie['admin_time'])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [40]:
# ── CHECK 1: patient overlap ───────────────────────────────────────────────
rx_hadm = set(rx['hadm_id'])
ie_hadm = set(ie['hadm_id'])

only_rx = rx_hadm - ie_hadm
only_ie = ie_hadm - rx_hadm
both    = rx_hadm & ie_hadm

print("\n" + "=" * 55)
print("CHECK 1: ADMISSION OVERLAP")
print("=" * 55)
print(f"  Prescriptions only : {len(only_rx):>6}  (non-ICU or uncharted)")
print(f"  Inputevents only   : {len(only_ie):>6}  (no matching RX order)")
print(f"  In BOTH            : {len(both):>6}")
print(f"  Overlap rate       : {len(both)/len(rx_hadm)*100:.1f}%")


CHECK 1: ADMISSION OVERLAP
  Prescriptions only :  56833  (non-ICU or uncharted)
  Inputevents only   :    497  (no matching RX order)
  In BOTH            :  28657
  Overlap rate       : 33.5%


Only 28,657 admissions appeared in both tables. The remaining 56,833 were prescriptions-only — mostly non-ICU patients who never had a pump charted in the ICU system.

## 3.3 the join

****

****

In [41]:
# ── 1. Load raw tables into DuckDB ─────────────────────────────────────────
# ── 1. Load raw tables into DuckDB ─────────────────────────────────────────
rx_raw = duckdb.query("""
    SELECT
        subject_id,
        hadm_id,
        TRY_CAST(dose_val_rx AS DOUBLE) AS dose_mg,
        CAST(starttime AS TIMESTAMP)    AS order_start,
        CAST(stoptime  AS TIMESTAMP)    AS order_stop
    FROM prescriptions
    WHERE drug        ILIKE '%vancomycin%'
      AND route        = 'IV'
      AND dose_unit_rx = 'mg'
      AND TRY_CAST(dose_val_rx AS DOUBLE) > 0
      AND starttime IS NOT NULL
      AND stoptime  IS NOT NULL
""").df()

ie_raw = duckdb.query("""
    SELECT
        subject_id,
        hadm_id,
        stay_id,
        CAST(starttime AS TIMESTAMP) AS admin_time,
        amount,
        amountuom,
        patientweight
    FROM inputevents
    WHERE itemid             = 225798
      AND statusdescription  = 'FinishedRunning'
      AND amount             > 0
""").df()

# Register for the join
duckdb.register('rx', rx_raw)
duckdb.register('ie', ie_raw)

# ── 2. Window join entirely in DuckDB with time_hours calculation ──────────
dose_events = duckdb.query("""
    SELECT
        ie.subject_id,
        ie.hadm_id,
        ie.stay_id,
        ie.admin_time,                          -- precise pump timestamp
        
        -- NEW: Calculate elapsed hours from ICU admission (intime)
        epoch(ie.admin_time - i.intime) / 3600.0 AS time_hours,
        
        rx.dose_mg AS valuminum,                
        0 AS event_type,                        -- static flag for doses
        
        rx.order_start,
        rx.order_stop,
        CASE WHEN ie.amountuom = 'mg' THEN ie.amount ELSE NULL END AS ie_amount_mg
    FROM ie ie
    
    -- NEW: Join with icustays to fetch the baseline ICU admission time
    JOIN icustays i
      ON ie.stay_id = i.stay_id
      
    JOIN rx rx
      ON  ie.subject_id  = rx.subject_id
      AND ie.hadm_id     = rx.hadm_id
      AND ie.admin_time >= rx.order_start
      AND ie.admin_time <= rx.order_stop
      
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY ie.subject_id, ie.hadm_id, ie.admin_time
        ORDER BY rx.order_start DESC           -- most recent order wins
    ) = 1
""").df()

# ── 3. Isolate the exact schema requested earlier ─────────────────────────────
df_dose = dose_events[[
    'subject_id', 
    'hadm_id', 
    'stay_id', 
    'time_hours', 
    'valuminum', 
    'event_type'
]].copy()

print(df_dose.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   subject_id   hadm_id   stay_id  time_hours  valuminum  event_type
0    10012552  27089790  33383124   43.009444     1000.0           0
1    10035168  25449821  39978210    7.033333     1000.0           0
2    10049041  22532814  38600445   19.321111      750.0           0
3    10069423  23555007  39415898   67.566667     1000.0           0
4    10104308  22532312  37317907   95.916667     1250.0           0


In [42]:
# ── 3. Sanity check ────────────────────────────────────────────────────────
print(f"Dose events      : {len(df_dose):,}")
print(f"Unique admissions: {df_dose['hadm_id'].nunique():,}")
print(f"Unique patients  : {df_dose['subject_id'].nunique():,}")
print(f"\ndose_mg stats:")
print(df_dose['valuminum'].describe(percentiles=[.05,.25,.5,.75,.95]).to_string())

Dose events      : 121,672
Unique admissions: 27,180
Unique patients  : 23,454

dose_mg stats:
count    121672.000000
mean       1055.560852
std         226.077258
min         100.000000
5%          750.000000
25%        1000.000000
50%        1000.000000
75%        1000.000000
95%        1500.000000
max        2000.000000


****

****

# 4. extract lab values from labevents

In [43]:
# ── Step 1: Define the cohort ─────────────────────────────────────────────────
# Only stays that appear in BOTH events (concentrations) AND dose_events (doses)
# dose_events is hadm_id-based, so we join through icustays to get stay_id

duckdb.register('df_concentration', df_concentration)
duckdb.register('dose_events', dose_events)

cohort = duckdb.query("""
    WITH conc_stays AS (
        SELECT DISTINCT stay_id FROM df_concentration          -- 24,027 stays with vanco levels
    ),
    dose_stays AS (
        SELECT DISTINCT i.stay_id
        FROM dose_events d
        JOIN icustays i 
          ON d.subject_id = i.subject_id 
         AND d.hadm_id    = i.hadm_id    -- bridge from hadm to stay
    )
    SELECT stay_id FROM conc_stays
    WHERE  stay_id IN (SELECT stay_id FROM dose_stays)
""").df()

print(f"Cohort stays (have both levels + doses): {len(cohort):,}")

Cohort stays (have both levels + doses): 18,993


In [44]:
# ── Step 2: Extract labs for cohort stays only ────────────────────────────────
duckdb.register('cohort', cohort)

query_labs = """
    WITH lab_base AS (
        SELECT
            i.stay_id,
            epoch(l.charttime - i.intime) / 3600.0 AS time_hours,
            l.valuenum,
            l.itemid
        FROM labs l
        JOIN icustays i 
          ON l.hadm_id = i.hadm_id
        WHERE l.itemid IN (
            50912, 52546,               -- Creatinine
            51006, 52647,               -- BUN
            50862, 52022, 53085, 53138, -- Albumin
            50861,                      -- ALT
            50878                       -- AST
        )
          AND l.valuenum  IS NOT NULL
          AND l.valuenum   > 0
          AND l.charttime >= i.intime
          AND l.charttime <= i.outtime
          AND i.stay_id IN (SELECT stay_id FROM cohort)   -- cohort filter
    )
    SELECT
        stay_id,
        time_hours,
        valuenum,
        CASE
            WHEN itemid IN (50912, 52546)               THEN 2  -- creatinine
            WHEN itemid IN (51006, 52647)               THEN 3  -- BUN
            WHEN itemid IN (50862, 52022, 53085, 53138) THEN 4  -- albumin
            WHEN itemid  = 50861                        THEN 5  -- ALT
            WHEN itemid  = 50878                        THEN 6  -- AST
        END AS event_type
    FROM lab_base
    ORDER BY stay_id, event_type, time_hours
"""

df_labs = duckdb.query(query_labs).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [45]:
# ── Step 3: Sanity check ──────────────────────────────────────────────────────
event_names = {2: 'creatinine', 3: 'BUN', 4: 'albumin', 5: 'ALT', 6: 'AST'}

print(f"Total lab rows : {len(df_labs):,}")
print(f"Unique stays   : {df_labs['stay_id'].nunique():,}")
print(f"\nBreakdown by event_type:")

summary = df_labs.groupby('event_type').agg(
    lab      = ('event_type', lambda x: event_names[x.iloc[0]]),
    n_rows   = ('valuenum',   'count'),
    n_stays  = ('stay_id',    'nunique'),
    median   = ('valuenum',   'median'),
    p5       = ('valuenum',   lambda x: x.quantile(.05)),
    p95      = ('valuenum',   lambda x: x.quantile(.95)),
)
print(summary.to_string())

print(f"\nSample rows:")
print(df_labs.head(10))

Total lab rows : 684,192
Unique stays   : 18,705

Breakdown by event_type:
                   lab  n_rows  n_stays  median    p5     p95
event_type                                                   
2           creatinine  248383    18699     1.2   0.4     4.7
3                  BUN  248014    18697    28.0   8.0    93.0
4              albumin   36245    11107     2.7   1.8     3.9
5                  ALT   75319    13812    43.0   9.0   974.0
6                  AST   76231    13921    59.0  15.0  1227.0

Sample rows:
    stay_id  time_hours  valuenum  event_type
0  30000484   11.074444       1.2           2
1  30000484   34.024444       1.3           2
2  30000484   58.574444       1.2           2
3  30000484   11.074444      47.0           3
4  30000484   34.024444      41.0           3
5  30000484   58.574444      36.0           3
6  30000484    3.957778       2.5           4
7  30000484   34.024444       2.8           4
8  30000484   58.574444       2.5           4
9  30000484   11.

In [46]:
df_labs.describe().T

,count,mean,std,min,25%,50%,75%,max
stay_id,684192.0,3.498874e+07,2.893510e+06,30000484.0,3.249059e+07,3.501124e+07,3.748148e+07,3.999923e+07
time_hours,684192.0,1.820955e+02,2.390675e+02,0.0,3.362806e+01,1.026833e+02,2.409749e+02,3.824143e+03
valuenum,684192.0,7.369677e+01,4.925795e+02,0.1,1.800000e+00,1.400000e+01,4.000000e+01,6.185400e+04
event_type,684192.0,3.244366e+00,1.341414e+00,2.0,2.000000e+00,3.000000e+00,4.000000e+00,6.000000e+00


****

# 5. covarities

In [47]:
df_labs.head()

,stay_id,time_hours,valuenum,event_type
0,30000484,11.074444,1.2,2
1,30000484,34.024444,1.3,2
2,30000484,58.574444,1.2,2
3,30000484,11.074444,47.0,3
4,30000484,34.024444,41.0,3


In [48]:
df_dose.head()

,subject_id,hadm_id,stay_id,time_hours,valuminum,event_type
0,10012552,27089790,33383124,43.009444,1000.0,0
1,10035168,25449821,39978210,7.033333,1000.0,0
2,10049041,22532814,38600445,19.321111,750.0,0
3,10069423,23555007,39415898,67.566667,1000.0,0
4,10104308,22532312,37317907,95.916667,1250.0,0


In [49]:
df_concentration.head()

,stay_id,time_hours,valuenum,event_type
0,37081114,131.800000,28.2,1
1,37081114,202.500000,20.9,1
2,37067082,29.949444,8.0,1
3,37067082,44.532778,11.9,1
4,34592300,61.610000,11.1,1


In [50]:
# 1. Extract unique stay_ids from each dataframe as Python sets
labs_stays = set(df_labs['stay_id'])
dose_stays = set(df_dose['stay_id'])
conc_stays = set(df_concentration['stay_id'])

# 2. Find the intersection (stay_ids present in ALL three sets)
common_stays = labs_stays.intersection(dose_stays).intersection(conc_stays)

# 3. Print the results
print("=" * 45)
print(f"Unique stays in df_labs         : {len(labs_stays):,}")
print(f"Unique stays in df_dose         : {len(dose_stays):,}")
print(f"Unique stays in df_concentration: {len(conc_stays):,}")
print("-" * 45)
print(f"Stays present in ALL 3 tables   : {len(common_stays):,}")
print("=" * 45)

Unique stays in df_labs         : 18,705
Unique stays in df_dose         : 28,912
Unique stays in df_concentration: 24,027
---------------------------------------------
Stays present in ALL 3 tables   : 16,460


In [51]:
duckdb.register('df_labs', df_labs)
duckdb.register('dose_events', df_dose)
duckdb.register('events', df_concentration)

final_cohort = duckdb.query("""
    WITH lab_stays  AS (SELECT DISTINCT stay_id FROM df_labs),
         conc_stays AS (SELECT DISTINCT stay_id FROM events),
         dose_stays AS (SELECT DISTINCT stay_id FROM dose_events
                        WHERE stay_id IS NOT NULL)
    SELECT stay_id FROM lab_stays
    WHERE  stay_id IN (SELECT stay_id FROM conc_stays)
      AND  stay_id IN (SELECT stay_id FROM dose_stays)
""").df()

print(f"Final cohort: {len(final_cohort):,} stays")  # should be 16,460
duckdb.register('final_cohort', final_cohort)

Final cohort: 16,460 stays


In [52]:
# ── Step 2: age + sex ─────────────────────────────────────────────────────────
covs_base = duckdb.query("""
    SELECT
        i.stay_id,
        i.subject_id,
        i.hadm_id,
        p.anchor_age          AS age,
        p.gender              AS sex,
        EPOCH(i.outtime - i.intime) / 3600.0 AS los_icu_hrs,
        i.intime,
        i.outtime
    FROM icustays i
    JOIN patients p ON i.subject_id = p.subject_id
    WHERE i.stay_id IN (SELECT stay_id FROM final_cohort)
""").df()

print(f"Base rows: {len(covs_base):,}")
print(covs_base[['age','sex']].describe(include='all').T)

Base rows: 16,460
       count unique  top  freq       mean        std   min   25%   50%   75%  \
age  16460.0    NaN  NaN   NaN  62.440279  15.936223  18.0  53.0  64.0  74.0   
sex    16460      2    M  9774        NaN        NaN   NaN   NaN   NaN   NaN   

      max  
age  91.0  
sex   NaN  


In [53]:
covs_base.describe().T

,count,mean,min,25%,50%,75%,max,std
stay_id,16460.0,34982753.490279,30000484.0,32490959.25,34971596.5,37469441.25,39999230.0,2889131.48899
subject_id,16460.0,14999854.171324,10000690.0,12502167.0,14998646.0,17517362.0,19999828.0,2884211.930867
hadm_id,16460.0,25003176.100182,20001305.0,22515835.0,25014192.0,27461540.25,29999098.0,2873997.059595
age,16460.0,62.440279,18.0,53.0,64.0,74.0,91.0,15.936223
los_icu_hrs,16460.0,193.953256,3.0975,59.576667,120.131806,248.753819,3832.003333,213.811101
intime,16460,2154-03-25 22:52:51.978190,2110-01-16 04:28:00,2134-06-20 05:35:26.250000,2154-01-31 03:24:00,2174-06-11 19:06:36.750000,2210-10-19 21:08:00,NaN
outtime,16460,2154-04-03 00:50:03.699757,2110-01-17 17:51:08,2134-06-25 11:37:07.750000,2154-02-10 03:42:25.500000,2174-06-24 03:39:32.500000,2210-11-02 17:45:11,NaN


In [54]:
# ── Step 3: weight + height ───────────────────────────────────────────────────
covs_anthropometrics = duckdb.query("""
    WITH ranked AS (
        SELECT
            c.stay_id,
            c.itemid,
            c.valuenum,
            ROW_NUMBER() OVER (
                PARTITION BY c.stay_id, c.itemid
                ORDER BY c.charttime
            ) AS rn
        FROM chartevents c
        WHERE c.stay_id  IN (SELECT stay_id FROM final_cohort)
          AND c.itemid   IN (226512, 226730)
          AND c.valuenum IS NOT NULL
          AND c.valuenum  > 0
    )
    SELECT
        stay_id,
        MAX(CASE WHEN itemid = 226512 THEN valuenum END) AS weight_kg,
        MAX(CASE WHEN itemid = 226730 THEN valuenum END) AS height_cm
    FROM ranked
    WHERE rn = 1
    GROUP BY stay_id
""").df()

print(f"Stays with weight: {covs_anthropometrics['weight_kg'].notna().sum():,}")
print(f"Stays with height: {covs_anthropometrics['height_cm'].notna().sum():,}")
print(covs_anthropometrics[['weight_kg','height_cm']].describe().T.round(1))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Stays with weight: 14,814
Stays with height: 10,109
             count   mean      std  min    25%    50%    75%        max
weight_kg  14814.0   83.9     27.5  1.0   66.4   80.0   96.4      710.4
height_cm  10109.0  376.5  20841.8  6.0  163.0  170.0  178.0  2095680.0


In [55]:
covs_anthropometrics.head()

,stay_id,weight_kg,height_cm
0,30858314,60.7,173.0
1,31124674,97.0,180.0
2,31699190,51.5,NaN
3,32979487,82.2,188.0
4,33016256,94.5,183.0


In [56]:
# ── Step 4: ventilation flags ─────────────────────────────────────────────────
covs_vent = duckdb.query("""
    SELECT
        stay_id,
        MAX(CASE WHEN itemid = 225792 THEN 1 ELSE 0 END) AS has_invasive_vent,
        MAX(CASE WHEN itemid = 225794 THEN 1 ELSE 0 END) AS has_non_invasive_vent
    FROM procedureevents
    WHERE stay_id IN (SELECT stay_id FROM final_cohort)
      AND itemid  IN (225792, 225794)
    GROUP BY stay_id
""").df()

print(f"Stays with invasive vent    : {covs_vent['has_invasive_vent'].sum():,}")
print(f"Stays with non-invasive vent: {covs_vent['has_non_invasive_vent'].sum():,}")

Stays with invasive vent    : 10,101
Stays with non-invasive vent: 1,019


In [57]:
# ── Step 5: dialysis flags ────────────────────────────────────────────────────
covs_dialysis = duckdb.query("""
    SELECT
        stay_id,
        MAX(CASE WHEN itemid = 225441                    THEN 1 ELSE 0 END) AS has_hd,
        MAX(CASE WHEN itemid IN (225802, 225803, 225809) THEN 1 ELSE 0 END) AS has_crrt,
        MAX(CASE WHEN itemid = 225805                    THEN 1 ELSE 0 END) AS has_pd,
        MAX(CASE WHEN itemid = 225955                    THEN 1 ELSE 0 END) AS has_other_dialysis
    FROM procedureevents
    WHERE stay_id IN (SELECT stay_id FROM final_cohort)
      AND itemid  IN (225441, 225802, 225803, 225809, 225805, 225955)
    GROUP BY stay_id
""").df()

print(f"Stays with HD              : {covs_dialysis['has_hd'].sum():,}")
print(f"Stays with CRRT            : {covs_dialysis['has_crrt'].sum():,}")
print(f"Stays with PD              : {covs_dialysis['has_pd'].sum():,}")
print(f"Stays with other dialysis  : {covs_dialysis['has_other_dialysis'].sum():,}")

Stays with HD              : 1,219
Stays with CRRT            : 1,591
Stays with PD              : 49
Stays with other dialysis  : 0


In [58]:
# ── Step 6: vasopressor flags ────────────────────────────────────────────────────
covs_vasopressor = duckdb.query("""
    SELECT stay_id, 1 AS has_vasopressor
    FROM   inputevents
    WHERE  itemid IN (221906, 221289, 221662, 222315, 221749)
    AND    statusdescription = 'FinishedRunning'
    AND    stay_id IN (SELECT stay_id FROM final_cohort)
    GROUP BY stay_id
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [59]:
covs_vasopressor.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6232 entries, 0 to 6231
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   stay_id          6232 non-null   int64
 1   has_vasopressor  6232 non-null   int32
dtypes: int32(1), int64(1)
memory usage: 73.2 KB


In [60]:
# ── Step 7: merge everything ──────────────────────────────────────────────────
covariates = (
    covs_base
    .merge(covs_anthropometrics, on='stay_id', how='left')
    .merge(covs_vent,            on='stay_id', how='left')
    .merge(covs_dialysis,        on='stay_id', how='left')
    .merge(covs_vasopressor,     on='stay_id', how='left')
)

flag_cols = ['has_invasive_vent', 'has_non_invasive_vent',
             'has_hd', 'has_crrt', 'has_pd', 'has_other_dialysis', 'has_vasopressor']
covariates[flag_cols] = covariates[flag_cols].fillna(0).astype(int)

# ── Sanity check ──────────────────────────────────────────────────────────────
print(f"Shape             : {covariates.shape}")
print(f"Unique stays      : {covariates['stay_id'].nunique():,}")
print(f"\nMissing values:")
print(covariates.isnull().sum())
print(f"\nBinary flag prevalence (%):")
print((covariates[flag_cols].mean() * 100).round(1))
print(f"\nContinuous covariate summary:")
print(covariates[['age', 'weight_kg', 'height_cm', 'los_icu_hrs']].describe().round(1).T)

Shape             : (16460, 17)
Unique stays      : 16,460

Missing values:
stay_id                     0
subject_id                  0
hadm_id                     0
age                         0
sex                         0
los_icu_hrs                 0
intime                      0
outtime                     0
weight_kg                1646
height_cm                6351
has_invasive_vent           0
has_non_invasive_vent       0
has_hd                      0
has_crrt                    0
has_pd                      0
has_other_dialysis          0
has_vasopressor             0
dtype: int64

Binary flag prevalence (%):
has_invasive_vent        61.4
has_non_invasive_vent     6.2
has_hd                    7.4
has_crrt                  9.7
has_pd                    0.3
has_other_dialysis        0.0
has_vasopressor          37.9
dtype: float64

Continuous covariate summary:
               count   mean      std   min    25%    50%    75%        max
age          16460.0   62.4     15.9  18.

In [61]:
covariates.head()

,stay_id,subject_id,hadm_id,age,sex,los_icu_hrs,intime,outtime,weight_kg,height_cm,has_invasive_vent,has_non_invasive_vent,has_hd,has_crrt,has_pd,has_other_dialysis,has_vasopressor
0,37330124,16650418,25388051,76,M,167.445000,2147-06-07 19:30:00,2147-06-14 18:56:42,73.0,175.0,1,1,0,0,0,0,0
1,39009171,16651008,24652180,40,F,26.317778,2145-09-09 22:52:00,2145-09-11 01:11:04,105.0,NaN,0,0,1,0,0,0,0
2,30290653,16652060,26645556,77,M,358.936111,2189-02-06 23:33:00,2189-02-21 22:29:10,66.4,180.0,1,1,0,0,0,0,1
3,35855170,16652205,28093183,85,M,478.578889,2169-03-12 23:14:02,2169-04-01 21:48:46,82.4,NaN,1,0,0,0,0,0,1
4,35105595,16654740,29903568,50,M,342.792500,2134-03-28 19:22:10,2134-04-12 02:09:43,126.2,140.0,1,0,0,0,0,0,1


In [62]:
covariates.describe().T

,count,mean,min,25%,50%,75%,max,std
stay_id,16460.0,34982753.490279,30000484.0,32490959.25,34971596.5,37469441.25,39999230.0,2889131.48899
subject_id,16460.0,14999854.171324,10000690.0,12502167.0,14998646.0,17517362.0,19999828.0,2884211.930867
hadm_id,16460.0,25003176.100182,20001305.0,22515835.0,25014192.0,27461540.25,29999098.0,2873997.059595
age,16460.0,62.440279,18.0,53.0,64.0,74.0,91.0,15.936223
los_icu_hrs,16460.0,193.953256,3.0975,59.576667,120.131806,248.753819,3832.003333,213.811101
intime,16460,2154-03-25 22:52:51.978190,2110-01-16 04:28:00,2134-06-20 05:35:26.250000,2154-01-31 03:24:00,2174-06-11 19:06:36.750000,2210-10-19 21:08:00,NaN
outtime,16460,2154-04-03 00:50:03.699757,2110-01-17 17:51:08,2134-06-25 11:37:07.750000,2154-02-10 03:42:25.500000,2174-06-24 03:39:32.500000,2210-11-02 17:45:11,NaN
weight_kg,14814.0,83.888315,1.0,66.425,80.0,96.4,710.4,27.497275
height_cm,10109.0,376.528283,6.0,163.0,170.0,178.0,2095680.0,20841.831343
has_invasive_vent,16460.0,0.61367,0.0,0.0,1.0,1.0,1.0,0.486923


# 6. save & upload

In [63]:
covariates.to_parquet('covariates.parquet', index=False)
df_concentration.to_parquet('df_concentration.parquet', index=False)
df_dose.to_parquet('df_dose.parquet', index=False)
df_labs.to_parquet('df_labs.parquet', index=False)


print(f"covariates.parquet : {len(covariates):,} rows  |  {os.path.getsize('covariates.parquet')/1e3:.0f} KB")
print(f"df_concentration.parquet : {len(df_concentration):,} rows  |  {os.path.getsize('df_concentration.parquet')/1e6:.1f} MB")
print(f"df_dose.parquet : {len(df_dose):,} rows  |  {os.path.getsize('df_dose.parquet')/1e6:.1f} MB")
print(f"df_labs.parquet : {len(df_labs):,} rows  |  {os.path.getsize('df_labs.parquet')/1e6:.1f} MB")

covariates.parquet : 16,460 rows  |  838 KB
df_concentration.parquet : 83,675 rows  |  1.0 MB
df_dose.parquet : 121,672 rows  |  1.9 MB
df_labs.parquet : 684,192 rows  |  3.5 MB
